# G04 - Hal Fiyatlarıyla Akıllı Satın Alma: Keşifsel Veri Analizi (EDA) ve Karar Destek Özeti (`02_eda.ipynb`)

Bu notebook; dağılmış olan fiyat analizi, günlük/haftalık/aylık/mevsimsel satın alma karar tabloları (`haftalik_fiyat_analizi.py` vb.), reel karar desteği (`01_Veri/ilk_25_reel_karar_destegi.csv`) ve 6 adet keşifsel veri analizi (EDA) grafiğini tek bir standart yapıda birleştirir.
- **`MAL_TURU` Standardizasyonu:** `İTHAL..` / `ITHAL..` ve `İTHAL` kayıtları analize ve modellemeye girmeden önce tek bir **`ITHAL`** kategorisinde birleştirilir.
- **Asgari (`ASGARI_UCRET`) ve Azami (`AZAMI_UCRET`) Fiyat Kontrolü:** Arayüzde fiyat bandı ve tedarikçi teklif kontrolü için Min/Max sütunlarının `hal_reel_ozellikli.csv` içinde korunduğu doğrulanır.

## 1. Kütüphaneler, Klasör Yolları ve Veri Doğrulama

In [ ]:
import re
from pathlib import Path
import matplotlib.pyplot as plt
import pandas as pd

def proje_koku_bul(baslangic: Path = Path.cwd()) -> Path:
    for aday in [baslangic] + list(baslangic.parents):
        if (aday / "01_Veri").exists():
            return aday
    return baslangic

PROJE_KOKU = proje_koku_bul()
VERI_KLASORU = PROJE_KOKU / "01_Veri"
RAPOR_KLASORU = PROJE_KOKU / "04_Rapor"
RAPOR_KLASORU.mkdir(parents=True, exist_ok=True)

def mal_turu_standartlastir(deger: str) -> str:
    s = str(deger).strip().upper()
    s = re.sub(r"\s+", " ", s)
    s_norm = s.replace("İ", "I").strip(".")
    if s_norm.startswith("ITHAL"):
        return "ITHAL"
    return s.strip(".")

df_ozellikli = pd.read_csv(VERI_KLASORU / "hal_reel_ozellikli.csv", low_memory=False)
df_ozellikli["MAL_TURU"] = df_ozellikli["MAL_TURU"].apply(mal_turu_standartlastir)
df_ozellikli["MAL_ADI"] = (
    df_ozellikli["MAL_ADI"].astype(str).str.strip().str.upper().str.replace(r"\s+", " ", regex=True)
)

assert "ASGARI_UCRET" in df_ozellikli.columns and "AZAMI_UCRET" in df_ozellikli.columns, (
    "HATA: ASGARI_UCRET (Min) ve AZAMI_UCRET (Max) sütunları hal_reel_ozellikli.csv içinde bulunmalıdır!"
)
print("Veri boyutu:", df_ozellikli.shape)
print("Min/Max sütun kontrolü başarılı: ASGARI_UCRET ve AZAMI_UCRET mevcut.")
print("Standartlaştırılmış MAL_TURU dağılımı:\n", df_ozellikli["MAL_TURU"].value_counts())

## 2. Genel, Günlük, Haftalık ve Aylık Satın Alma Karar Analizleri (`fiyat_analizi`, `gunluk_fiyat_analizi`, `haftalik_fiyat_analizi`, `aylik_fiyat_karar`, `uygun_ay_ozeti`)

In [ ]:
df_temiz = pd.read_csv(VERI_KLASORU / "temiz_hal_fiyatlari.csv", encoding="utf-8-sig", low_memory=False)
df_temiz["TARIH"] = pd.to_datetime(df_temiz["TARIH"], errors="coerce")
df_temiz["MAL_TURU"] = df_temiz["MAL_TURU"].apply(mal_turu_standartlastir)
df_temiz["MAL_ADI"] = df_temiz["MAL_ADI"].astype(str).str.strip().str.upper().str.replace(r"\s+", " ", regex=True)

# 2.1 Ürün Fiyat Özeti
urun_ozeti = (
    df_temiz.groupby("MAL_ADI")
    .agg(
        KAYIT_SAYISI=("ORTALAMA_UCRET", "count"),
        MIN_ORT_FIYAT=("ORTALAMA_UCRET", "min"),
        ORTALAMA_FIYAT=("ORTALAMA_UCRET", "mean"),
        MAX_ORT_FIYAT=("ORTALAMA_UCRET", "max"),
    )
    .sort_values("KAYIT_SAYISI", ascending=False)
    .reset_index()
)
urun_ozeti["ORTALAMA_FIYAT"] = urun_ozeti["ORTALAMA_FIYAT"].round(2)
urun_ozeti.to_csv(VERI_KLASORU / "urun_fiyat_ozeti.csv", index=False, encoding="utf-8-sig")

ilk_25_listesi = (
    pd.read_csv(VERI_KLASORU / "ilk_25_urun.csv", encoding="utf-8-sig")["MAL_ADI"]
    .astype(str).str.strip().str.upper().str.replace(r"\s+", " ", regex=True).tolist()
)
df_25 = df_temiz[df_temiz["MAL_ADI"].isin(ilk_25_listesi)].copy()

genel_ort = (
    df_25.groupby("MAL_ADI")["ORTALAMA_UCRET"]
    .mean()
    .round(2)
    .reset_index(name="GENEL_ORT_FIYAT")
)

# 2.2 Günlük Satın Alma Karar Tablosu
df_25["GUN"] = df_25["TARIH"].dt.date
gunluk = (
    df_25.groupby(["MAL_ADI", "GUN"])["ORTALAMA_UCRET"]
    .mean()
    .round(2)
    .reset_index(name="GUNLUK_ORT_FIYAT")
)
gunluk_karar = gunluk.merge(genel_ort, on="MAL_ADI")
gunluk_karar["DURUM"] = gunluk_karar.apply(
    lambda r: "UYGUN GUN" if r["GUNLUK_ORT_FIYAT"] < r["GENEL_ORT_FIYAT"] * 0.9
    else ("PAHALI GUN" if r["GUNLUK_ORT_FIYAT"] > r["GENEL_ORT_FIYAT"] * 1.1 else "NORMAL"),
    axis=1,
)
gunluk_karar.to_csv(VERI_KLASORU / "gunluk_satinalma_karar.csv", index=False, encoding="utf-8-sig")

# 2.3 Haftalık Satın Alma Karar Tablosu (haftalik_fiyat_analizi.py)
df_25["HAFTA"] = df_25["TARIH"].dt.to_period("W").apply(lambda r: r.start_time.date())
haftalik = (
    df_25.groupby(["MAL_ADI", "HAFTA"])["ORTALAMA_UCRET"]
    .mean()
    .round(2)
    .reset_index(name="HAFTALIK_ORT_FIYAT")
)
haftalik_karar = haftalik.merge(genel_ort, on="MAL_ADI")
haftalik_karar["DURUM"] = haftalik_karar.apply(
    lambda r: "UYGUN HAFTA" if r["HAFTALIK_ORT_FIYAT"] < r["GENEL_ORT_FIYAT"] * 0.9
    else ("PAHALI HAFTA" if r["HAFTALIK_ORT_FIYAT"] > r["GENEL_ORT_FIYAT"] * 1.1 else "NORMAL"),
    axis=1,
)
haftalik_karar.to_csv(VERI_KLASORU / "haftalik_satinalma_karar.csv", index=False, encoding="utf-8-sig")

# 2.4 Aylık Satın Alma Karar Tablosu ve Uygun Ay Özeti
df_25["AY"] = df_25["TARIH"].dt.month
aylik = (
    df_25.groupby(["MAL_ADI", "AY"])["ORTALAMA_UCRET"]
    .mean()
    .round(2)
    .reset_index(name="AYLIK_ORT_FIYAT")
)
aylik_karar = aylik.merge(genel_ort, on="MAL_ADI")
aylik_karar["DURUM"] = aylik_karar.apply(
    lambda r: "UYGUN DONEM" if r["AYLIK_ORT_FIYAT"] < r["GENEL_ORT_FIYAT"] * 0.9
    else ("PAHALI DONEM" if r["AYLIK_ORT_FIYAT"] > r["GENEL_ORT_FIYAT"] * 1.1 else "NORMAL"),
    axis=1,
)
aylik_karar.to_csv(VERI_KLASORU / "aylik_satinalma_karar.csv", index=False, encoding="utf-8-sig")

ozet_satirlari = []
for urun, grup in aylik_karar.groupby("MAL_ADI"):
    uygun_aylar = grup[grup["DURUM"] == "UYGUN DONEM"]["AY"].sort_values().astype(str).tolist()
    pahali_aylar = grup[grup["DURUM"] == "PAHALI DONEM"]["AY"].sort_values().astype(str).tolist()
    en_ucuz_satir = grup.loc[grup["AYLIK_ORT_FIYAT"].idxmin()]
    en_pahali_satir = grup.loc[grup["AYLIK_ORT_FIYAT"].idxmax()]
    ozet_satirlari.append({
        "MAL_ADI": urun,
        "GENEL_ORT_FIYAT": en_ucuz_satir["GENEL_ORT_FIYAT"],
        "EN_UCUZ_AY": int(en_ucuz_satir["AY"]),
        "EN_UCUZ_FIYAT": en_ucuz_satir["AYLIK_ORT_FIYAT"],
        "EN_PAHALI_AY": int(en_pahali_satir["AY"]),
        "EN_PAHALI_FIYAT": en_pahali_satir["AYLIK_ORT_FIYAT"],
        "UYGUN_AYLAR": ", ".join(uygun_aylar) if uygun_aylar else "-",
        "PAHALI_AYLAR": ", ".join(pahali_aylar) if pahali_aylar else "-",
    })
ozet_df = pd.DataFrame(ozet_satirlari).sort_values("MAL_ADI")
ozet_df.to_csv(VERI_KLASORU / "urun_uygun_ay_ozeti.csv", index=False, encoding="utf-8-sig")
print("Ürün bazlı uygun/pahalı ay özeti kaydedildi. Satır sayısı:", len(ozet_df))

## 3. Mevsimsel Reel Fiyat Analizi ve Reel Karar Desteği (`mevsimsel_fiyat_analizi.py` & `reel_karar_destegi.py`)

In [ ]:
df_reel = pd.read_csv(VERI_KLASORU / "hal_reel_fiyatli.csv", low_memory=False)
if "YIL" not in df_reel.columns or "AY" not in df_reel.columns:
    tarih_dt = pd.to_datetime(df_reel["TARIH"], errors="coerce")
    df_reel["YIL"] = tarih_dt.dt.year
    df_reel["AY"] = tarih_dt.dt.month

df_reel["MAL_TURU"] = df_reel["MAL_TURU"].apply(mal_turu_standartlastir)
for col in ["MAL_ADI", "BIRIM"]:
    if col in df_reel.columns:
        df_reel[col] = df_reel[col].astype(str).str.strip().str.upper().str.replace(r"\s+", " ", regex=True)

mevsim_haritasi = {
    12: "Kış", 1: "Kış", 2: "Kış",
    3: "İlkbahar", 4: "İlkbahar", 5: "İlkbahar",
    6: "Yaz", 7: "Yaz", 8: "Yaz",
    9: "Sonbahar", 10: "Sonbahar", 11: "Sonbahar",
}
ay_adlari = {
    1: "Ocak", 2: "Şubat", 3: "Mart", 4: "Nisan",
    5: "Mayıs", 6: "Haziran", 7: "Temmuz", 8: "Ağustos",
    9: "Eylül", 10: "Ekim", 11: "Kasım", 12: "Aralık",
}

df_reel["MEVSIM"] = df_reel["AY"].map(mevsim_haritasi)
df_reel["AY_ADI"] = df_reel["AY"].map(ay_adlari)

aylik_ozet = (
    df_reel.groupby(["MAL_ADI", "BIRIM", "AY", "AY_ADI", "MEVSIM"], as_index=False)["REEL_ORTALAMA_UCRET"]
    .mean()
    .round(2)
    .rename(columns={"REEL_ORTALAMA_UCRET": "AYLIK_REEL_ORT"})
)
aylik_ozet.to_csv(VERI_KLASORU / "ilk_25_aylik_reel_fiyat.csv", index=False)

mevsim_pivot = (
    df_reel.pivot_table(
        index=["MAL_ADI", "BIRIM"],
        columns="MEVSIM",
        values="REEL_ORTALAMA_UCRET",
        aggfunc="mean",
    )
    .round(2)
)
mevsim_sirasi = [m for m in ["Kış", "İlkbahar", "Yaz", "Sonbahar"] if m in mevsim_pivot.columns]
mevsim_pivot = mevsim_pivot[mevsim_sirasi]

en_ucuz_ay = (
    aylik_ozet.sort_values(["AYLIK_REEL_ORT", "AY"])
    .groupby(["MAL_ADI", "BIRIM"], as_index=False)
    .first()[["MAL_ADI", "BIRIM", "AY_ADI", "AYLIK_REEL_ORT"]]
    .rename(columns={"AY_ADI": "EN_UCUZ_AY", "AYLIK_REEL_ORT": "EN_UCUZ_REEL"})
)
en_pahali_ay = (
    aylik_ozet.sort_values(["AYLIK_REEL_ORT", "AY"], ascending=[False, True])
    .groupby(["MAL_ADI", "BIRIM"], as_index=False)
    .first()[["MAL_ADI", "BIRIM", "AY_ADI", "AYLIK_REEL_ORT"]]
    .rename(columns={"AY_ADI": "EN_PAHALI_AY", "AYLIK_REEL_ORT": "EN_PAHALI_REEL"})
)

mevsim_ozet = (
    mevsim_pivot.reset_index()
    .merge(en_ucuz_ay, on=["MAL_ADI", "BIRIM"], how="left")
    .merge(en_pahali_ay, on=["MAL_ADI", "BIRIM"], how="left")
)
mevsim_ozet["MEVSIMSEL_FARK"] = (mevsim_ozet["EN_PAHALI_REEL"] - mevsim_ozet["EN_UCUZ_REEL"]).round(2)
mevsim_ozet = mevsim_ozet.sort_values("MEVSIMSEL_FARK", ascending=False).reset_index(drop=True)
mevsim_ozet.to_csv(VERI_KLASORU / "ilk_25_mevsimsel_reel_ozet.csv", index=False)

# Reel Karar Desteği Tablosu -> 01_Veri/ilk_25_reel_karar_destegi.csv
mevsim_kolonlari = [m for m in ["Kış", "İlkbahar", "Yaz", "Sonbahar"] if m in mevsim_ozet.columns]
mevsim_uzun = mevsim_ozet.melt(
    id_vars=["MAL_ADI", "BIRIM"],
    value_vars=mevsim_kolonlari,
    var_name="MEVSIM",
    value_name="MEVSIM_REEL_ORT",
).dropna(subset=["MEVSIM_REEL_ORT"])

en_ucuz_mevsim = (
    mevsim_uzun.sort_values(["MEVSIM_REEL_ORT", "MEVSIM"])
    .groupby(["MAL_ADI", "BIRIM"], as_index=False)
    .first()[["MAL_ADI", "BIRIM", "MEVSIM", "MEVSIM_REEL_ORT"]]
    .rename(columns={"MEVSIM": "EN_UYGUN_MEVSIM", "MEVSIM_REEL_ORT": "EN_UYGUN_MEVSIM_ORT"})
)
en_pahali_mevsim = (
    mevsim_uzun.sort_values(["MEVSIM_REEL_ORT", "MEVSIM"], ascending=[False, True])
    .groupby(["MAL_ADI", "BIRIM"], as_index=False)
    .first()[["MAL_ADI", "BIRIM", "MEVSIM", "MEVSIM_REEL_ORT"]]
    .rename(columns={"MEVSIM": "EN_PAHALI_MEVSIM", "MEVSIM_REEL_ORT": "EN_PAHALI_REEL_ORT"})
)
yillik_reel_ort = (
    aylik_ozet.groupby(["MAL_ADI", "BIRIM"], as_index=False)["AYLIK_REEL_ORT"]
    .mean()
    .round(2)
    .rename(columns={"AYLIK_REEL_ORT": "YILLIK_REEL_ORT"})
)

karar_df = (
    mevsim_ozet[["MAL_ADI", "BIRIM", "EN_UCUZ_AY", "EN_UCUZ_REEL", "EN_PAHALI_AY", "EN_PAHALI_REEL", "MEVSIMSEL_FARK"]]
    .merge(en_ucuz_mevsim, on=["MAL_ADI", "BIRIM"], how="left")
    .merge(en_pahali_mevsim, on=["MAL_ADI", "BIRIM"], how="left")
    .merge(yillik_reel_ort, on=["MAL_ADI", "BIRIM"], how="left")
)
karar_df["FARK_YUZDE"] = ((karar_df["MEVSIMSEL_FARK"] / karar_df["EN_UCUZ_REEL"]) * 100).round(1)
karar_df["KARAR_NOTU"] = (
    karar_df["EN_UCUZ_AY"].astype(str) + " (" + karar_df["EN_UYGUN_MEVSIM"].astype(str)
    + ") alim icin uygun; " + karar_df["EN_PAHALI_AY"].astype(str)
    + " (" + karar_df["EN_PAHALI_MEVSIM"].astype(str) + ") en yuksek reel seviye"
)
karar_df = karar_df.sort_values("MEVSIMSEL_FARK", ascending=False).reset_index(drop=True)
karar_df.to_csv(VERI_KLASORU / "ilk_25_reel_karar_destegi.csv", index=False)
print("Karar destek tablosu 01_Veri/ilk_25_reel_karar_destegi.csv konumuna kaydedildi.")
print(karar_df.head(10).to_string(index=False))

## 4. Keşifsel Veri Analizi (EDA) Grafikleri (6 Grafik)
Grafikler hem notebook çıktısında görüntülenir hem de raporlama için `04_Rapor/` dizinine kaydedilir.

### Grafik 1: Dönemsel Ortalama Fiyat – Nominal vs. Reel (`eda_grafik_1.py`)
- **Ne görüyoruz / İş için ne anlama geliyor:** Nominal fiyatlar enflasyon etkisiyle keskin biçimde yükselirken, TÜFE ile arındırılmış reel fiyatlar mevsimsel dalgalanma bandında hareket etmektedir. Satın alma kararlarında nominal fiyat yerine reel fiyat baz alınmalıdır.

In [ ]:
df = df_ozellikli.copy()
df["TARIH_DT"] = pd.to_datetime(df["TARIH"], errors="coerce")
df = df.dropna(subset=["TARIH_DT", "REEL_ORTALAMA_UCRET"]).copy()
df["DONEM"] = df["TARIH_DT"].dt.to_period("M").dt.to_timestamp()

ozet = df.groupby("DONEM", as_index=False)[["ORTALAMA_UCRET", "REEL_ORTALAMA_UCRET"]].mean()

plt.figure(figsize=(10, 5))
plt.plot(ozet["DONEM"], ozet["ORTALAMA_UCRET"], label="Nominal Ortalama", linewidth=2)
plt.plot(ozet["DONEM"], ozet["REEL_ORTALAMA_UCRET"], label="Reel Ortalama (TÜFE Bazlı)", linewidth=2)
plt.title("Dönemsel Ortalama Fiyat: Nominal vs. Reel")
plt.xlabel("Dönem")
plt.ylabel("Fiyat (TL)")
plt.legend()
plt.grid(True, alpha=0.3)
plt.tight_layout()
plt.savefig(RAPOR_KLASORU / "grafik_1_nominal_vs_reel.png", dpi=150)
plt.show()

### Grafik 2: En Yüksek Reel Fiyat Oynaklığına Sahip İlk 10 Ürün (`eda_grafik_2.py`)
- **Ne görüyoruz / İş için ne anlama geliyor:** 30 günlük reel oynaklığı en yüksek ürünlerde spot alım riski yüksektir; restoran zinciri bu ürünlerde sözleşmeli alım veya uygun sezonda stoklama tercih etmelidir.

In [ ]:
oynaklik_kolonu = "REEL_OYNAKLIK_30" if "REEL_OYNAKLIK_30" in df.columns else "REEL_OYNAKLIK_7"
en_oynak = (
    df.dropna(subset=["MAL_ADI", oynaklik_kolonu])
    .groupby("MAL_ADI", as_index=False)[oynaklik_kolonu]
    .mean()
    .sort_values(oynaklik_kolonu, ascending=False)
    .head(10)
    .sort_values(oynaklik_kolonu, ascending=True)
)

plt.figure(figsize=(10, 5))
plt.barh(en_oynak["MAL_ADI"], en_oynak[oynaklik_kolonu])
plt.title(f"En Yüksek Reel Fiyat Oynaklığına Sahip İlk 10 Ürün ({oynaklik_kolonu})")
plt.xlabel("Ortalama Oynaklık Değeri")
plt.ylabel("Ürün")
plt.grid(True, axis="x", alpha=0.3)
plt.tight_layout()
plt.savefig(RAPOR_KLASORU / "grafik_2_en_oynak_urunler.png", dpi=150)
plt.show()

### Grafik 3: Aylara Göre Ortalama Reel Fiyat – Mevsimsel Profil (`eda_grafik_3.py`)
- **Ne görüyoruz / İş için ne anlama geliyor:** Yaz ve sonbahar başında hasat bolluğu nedeniyle ortalama reel fiyatlar düşerken kış aylarında sera ve depolama maliyetleriyle reel fiyatlar yükselmektedir.

In [ ]:
ay_isimleri = {1: "Oca", 2: "Şub", 3: "Mar", 4: "Nis", 5: "May", 6: "Haz", 7: "Tem", 8: "Ağu", 9: "Eyl", 10: "Eki", 11: "Kas", 12: "Ara"}
df["AY"] = df["TARIH_DT"].dt.month
aylik = df.groupby("AY", as_index=False)["REEL_ORTALAMA_UCRET"].mean().sort_values("AY")
aylik["AY_ADI"] = aylik["AY"].map(ay_isimleri)

plt.figure(figsize=(10, 5))
plt.plot(aylik["AY_ADI"], aylik["REEL_ORTALAMA_UCRET"], marker="o", linewidth=2)
plt.title("Aylara Göre Ortalama Reel Fiyat (Mevsimsel Profil)")
plt.xlabel("Ay")
plt.ylabel("Ortalama Reel Fiyat")
plt.grid(True, alpha=0.3)
plt.tight_layout()
plt.savefig(RAPOR_KLASORU / "grafik_3_aylik_mevsimsel_reel.png", dpi=150)
plt.show()

### Grafik 4: Mal Türüne Göre Ortalama Reel Fiyat – `ITHAL` Birleştirilmiş (`eda_grafik_4.py`)
- **Ne görüyoruz / İş için ne anlama geliyor:** `İTHAL..` ve `İTHAL` etiketleri tek bir `ITHAL` kategorisinde birleştirildiğinde ithal ürünlerin yerli sebze ve meyvelere kıyasla belirgin şekilde daha yüksek reel birim maliyete sahip olduğu görülmektedir.

In [ ]:
tur_ozet = (
    df.dropna(subset=["MAL_TURU", "REEL_ORTALAMA_UCRET"])
    .groupby("MAL_TURU", as_index=False)["REEL_ORTALAMA_UCRET"]
    .mean()
    .sort_values("REEL_ORTALAMA_UCRET", ascending=True)
)

plt.figure(figsize=(10, 5))
plt.barh(tur_ozet["MAL_TURU"], tur_ozet["REEL_ORTALAMA_UCRET"])
plt.title("Mal Türüne Göre Ortalama Reel Fiyat (ITHAL Birleştirilmiş)")
plt.xlabel("Ortalama Reel Fiyat")
plt.ylabel("Mal Türü")
plt.grid(True, axis="x", alpha=0.3)
plt.tight_layout()
plt.savefig(RAPOR_KLASORU / "grafik_4_mal_turu_reel_fiyat.png", dpi=150)
plt.show()

### Grafik 5: Ortalama Reel Fiyatı En Yüksek İlk 10 Ürün (`eda_grafik_5.py`)
- **Ne görüyoruz / İş için ne anlama geliyor:** Birim reel maliyeti en yüksek ürünler bütçe içinde en büyük ağırlığı oluşturduğundan, satın alma zamanlaması optimizasyonunda önceliklendirilmelidir.

In [ ]:
en_pahali = (
    df.dropna(subset=["MAL_ADI", "REEL_ORTALAMA_UCRET"])
    .groupby("MAL_ADI", as_index=False)["REEL_ORTALAMA_UCRET"]
    .mean()
    .sort_values("REEL_ORTALAMA_UCRET", ascending=False)
    .head(10)
    .sort_values("REEL_ORTALAMA_UCRET", ascending=True)
)

plt.figure(figsize=(10, 5))
plt.barh(en_pahali["MAL_ADI"], en_pahali["REEL_ORTALAMA_UCRET"])
plt.title("Ortalama Reel Fiyatı En Yüksek İlk 10 Ürün")
plt.xlabel("Ortalama Reel Fiyat")
plt.ylabel("Ürün")
plt.grid(True, axis="x", alpha=0.3)
plt.tight_layout()
plt.savefig(RAPOR_KLASORU / "grafik_5_en_yuksek_reel_fiyat_urunler.png", dpi=150)
plt.show()

### Grafik 6: Dönemsel Reel Fiyat Değişim Yüzdesi Dağılımı (`eda_grafik_6.py`)
- **Ne görüyoruz / İş için ne anlama geliyor:** Günlük/dönemsel reel fiyat değişimleri çoğunlukla %0 etrafında simetrik toplanırken, sağ kuyruktaki ani sıçramalar mevsim geçişleri ve arz şoklarını (don/kuraklık) işaret etmektedir.

In [ ]:
degisim = df["REEL_DEGISIM_YUZDE"].dropna()
alt, ust = degisim.quantile(0.01), degisim.quantile(0.99)
filtreli = degisim[(degisim >= alt) & (degisim <= ust)]

plt.figure(figsize=(10, 5))
plt.hist(filtreli, bins=40)
plt.axvline(0, linestyle="--", linewidth=1.5)
plt.title("Dönemsel Reel Fiyat Değişim Yüzdesi Dağılımı (%1-%99 Aralığı)")
plt.xlabel("Reel Fiyat Değişimi (%)")
plt.ylabel("Kayıt Sayısı")
plt.grid(True, alpha=0.3)
plt.tight_layout()
plt.savefig(RAPOR_KLASORU / "grafik_6_reel_degisim_dagilimi.png", dpi=150)
plt.show()

## 5. Özgün Araştırma Sorusu ve İstatistiksel Hipotez Testleri (Kılavuz Özgünlük Kriteri)
- **Özgün Araştırma Sorusu:** *"2006–2025 döneminde enflasyondan arındırıldığında reel olarak en çok pahalanan ve ucuzlayan ürünler hangileridir? Mevsimler ve mal türleri (`ITHAL` vs. yerli) arasındaki reel fiyat farkları istatistiksel olarak anlamlı mıdır?"*
- **Hipotez 1 ($H_1$):** Temel hal ürünlerinin reel fiyatları mevsimlere göre istatistiksel olarak anlamlı farklılık göstermektedir (Kruskal-Wallis testi, $\alpha = 0.05$).
- **Hipotez 2 ($H_1$):** `ITHAL`, `MEYVE` ve `SEBZE` kategorileri arasındaki reel fiyat düzeyleri istatistiksel olarak anlamlı biçimde farklıdır (Kruskal-Wallis testi, $\alpha = 0.05$).

In [ ]:
from scipy import stats

df_hip = df.dropna(subset=["TARIH_DT", "REEL_ORTALAMA_UCRET"]).copy()
df_hip["YIL"] = df_hip["TARIH_DT"].dt.year
df_hip["AY"] = df_hip["TARIH_DT"].dt.month
mevsim_h = {12: "Kış", 1: "Kış", 2: "Kış", 3: "İlkbahar", 4: "İlkbahar", 5: "İlkbahar", 6: "Yaz", 7: "Yaz", 8: "Yaz", 9: "Sonbahar", 10: "Sonbahar", 11: "Sonbahar"}
df_hip["MEVSIM"] = df_hip["AY"].map(mevsim_h)

# Hipotez 1: Mevsimler arası reel fiyat farkı (Kruskal-Wallis)
mev_gruplar = [g["REEL_ORTALAMA_UCRET"].values for _, g in df_hip.groupby("MEVSIM")]
h1_stat, p1_val = stats.kruskal(*mev_gruplar)
print(f"Hipotez 1 (Mevsimsellik) -> Kruskal-Wallis H = {h1_stat:.2f}, p = {p1_val:.4e} -> {'H1 KABUL (Anlamlı)' if p1_val < 0.05 else 'H0 Reddedilemedi'}")

# Hipotez 2: Mal türü (ITHAL vs MEYVE vs SEBZE) arası reel fiyat farkı
tur_gruplar = [g["REEL_ORTALAMA_UCRET"].values for _, g in df_hip.groupby("MAL_TURU")]
h2_stat, p2_val = stats.kruskal(*tur_gruplar)
print(f"Hipotez 2 (Mal Türü)     -> Kruskal-Wallis H = {h2_stat:.2f}, p = {p2_val:.4e} -> {'H1 KABUL (Anlamlı)' if p2_val < 0.05 else 'H0 Reddedilemedi'}")

# Özgün Soru: 2006-2010 vs 2021-2025 Reel Olarak En Çok Pahalanan / Ucuzlayan Ürünler
ilk_5 = df_hip[df_hip["YIL"] <= 2010].groupby("MAL_ADI", as_index=False)["REEL_ORTALAMA_UCRET"].mean().rename(columns={"REEL_ORTALAMA_UCRET": "ILK_5_YIL_REEL"})
son_5 = df_hip[df_hip["YIL"] >= 2021].groupby("MAL_ADI", as_index=False)["REEL_ORTALAMA_UCRET"].mean().rename(columns={"REEL_ORTALAMA_UCRET": "SON_5_YIL_REEL"})
reel_fark = ilk_5.merge(son_5, on="MAL_ADI", how="inner")
reel_fark["REEL_DEGISIM_YUZDE"] = (((reel_fark["SON_5_YIL_REEL"] - reel_fark["ILK_5_YIL_REEL"]) / reel_fark["ILK_5_YIL_REEL"]) * 100).round(2)
reel_fark = reel_fark.sort_values("REEL_DEGISIM_YUZDE", ascending=False)
print("\n2006-2010 -> 2021-2025 Reel Olarak En Çok Pahalanan İlk 5 Ürün:")
print(reel_fark.head(5).to_string(index=False))
print("\n2006-2010 -> 2021-2025 Reel Olarak En Çok Ucuzlayan İlk 5 Ürün:")
print(reel_fark.tail(5).to_string(index=False))